# Understand the dataset before modeling

Run the pipeline and dashboard export first. See [methodology](../docs/methodology.md) for protocol v2.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype'
ROOT = Path.cwd() if (Path.cwd()/'results').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/'src'))
experiment = json.loads((ROOT/'results/experiment.json').read_text())
assert experiment['protocol_version'] == 2


## Which files and measurements are actually present?

In [2]:
inventory = pd.read_csv(ROOT/'results/source_inventory.csv')
dictionary = pd.read_csv(ROOT/'results/data_dictionary.csv')
display(inventory)
dictionary.drop_duplicates('column').groupby(['kind','unit']).size().to_frame('channels')

,file,bytes,checksum,downloaded
0,2018_Leakages.csv,6234468,md5:c2c5fab90420da44f02e29775050abfe,True
1,L-TOWN.inp,412200,md5:bfbb16b8b463e0576200ba6ad360f68e,True
2,2019_Leakages.csv,10414660,md5:e1f0a43683813a90a9fec8562dde599b,True
3,2018_SCADA_Levels.csv,2721000,md5:92a512d6749ddf66fd6d99f2bdde6cc4,True
4,2018_SCADA.xlsx,92174099,md5:20c1224db8e379e31eba301d852e5cf7,False
5,2018_SCADA_Pressures.csv,22593812,md5:d389d8541350c19ff0bfc6b80f246d35,True
6,2019_SCADA.xlsx,92348474,md5:80242d5f59d39d0ef2306e2351c28367,False
7,2019_SCADA_Levels.csv,2720780,md5:e5a8050bd38729e4b7648b9d52abd5b1,True
8,2019_SCADA_Flows.csv,4130511,md5:28fc99fdcbf80fcd26079e7fe602d6dc,True
9,2019_SCADA_Pressures.csv,22606417,md5:5ea1e46d3f2f0a89a3f98d6fd39a851d,True


,,channels
kind,unit,
demand,L/h,82
flow,m3/h,3
level,m,1
pressure,m,33


## Are the timestamp grid and values complete?

In [3]:
audit = json.loads((ROOT/'results/data_audit.json').read_text())
quality = pd.DataFrame({k:v for k,v in audit.items() if '_SCADA_' in k}).T
quality[['rows','sensors','duplicate_timestamps','missing_timestamps','original_missing_cells','filled_cells']]

,rows,sensors,duplicate_timestamps,missing_timestamps,original_missing_cells,filled_cells
2018_SCADA_Pressures.csv,105120,33,0,0,0,0
2018_SCADA_Flows.csv,105120,3,0,0,0,0
2018_SCADA_Levels.csv,105120,1,0,0,0,0
2018_SCADA_Demands.csv,105120,82,0,0,0,0
2019_SCADA_Pressures.csv,105120,33,0,0,0,0
2019_SCADA_Flows.csv,105120,3,0,0,0,0
2019_SCADA_Levels.csv,105120,1,0,0,0,0
2019_SCADA_Demands.csv,105120,82,0,0,0,0


## Why is ordinary binary leak-presence classification unsuitable?

In [4]:
presence = {year: audit[f'{year}_Leakages.csv']['any_leak_fraction'] for year in [2018,2019]}
events = pd.read_csv(ROOT/'results/events.csv', parse_dates=['start','end'])
display(pd.Series(presence, name='fraction_with_any_leak'))
events.assign(year=events.start.dt.year).groupby(['year','type']).size().unstack(fill_value=0)

2018    0.978025
2019    1.000000
Name: fraction_with_any_leak, dtype: float64

type,abrupt,incipient
year,,
2018,6,8
2019,9,10


## Check nominal topology without hidden generation parameters

In [5]:
topology = json.loads((ROOT/'results/topology.json').read_text())
print(f"Nominal coordinates: {len(topology['nodes'])}; links: {len(topology['edges'])}")
print('Unspecified source timezone: retain naive wall-clock timestamps')

Nominal coordinates: 785; links: 909
Unspecified source timezone: retain naive wall-clock timestamps


## Interpretation and limits

The inspected SCADA files require no imputation. Persistent leaks make every 2019 timestamp leak-positive. The detector therefore targets new onsets, and configuration/pipe leak flows remain retrospective truth.